In [1]:
import networkx as nx
import itertools
import numpy as np
import time

In [2]:
def three_profile_vertex_primitive(g: nx.Graph):
    vertex_profile = {}
    for v in g.nodes:
        vertex_profile[v] = vertex_3_data()
    
    for x,y,z in (itertools.combinations(g.nodes, 3)):
        if g.has_edge(x,y) and g.has_edge(x,z) and g.has_edge(y,z):
            vertex_profile[x].num_triangles += 1
            vertex_profile[y].num_triangles += 1
            vertex_profile[z].num_triangles += 1
        elif g.has_edge(x,y) and g.has_edge(x,z) and not g.has_edge(y,z):
            vertex_profile[x].num_wedge_center += 1
            vertex_profile[y].num_wedge_leaf += 1
            vertex_profile[z].num_wedge_leaf += 1
        elif g.has_edge(x,y) and not g.has_edge(x,z) and g.has_edge(y,z):
            vertex_profile[y].num_wedge_center += 1
            vertex_profile[x].num_wedge_leaf += 1
            vertex_profile[z].num_wedge_leaf += 1
        elif not g.has_edge(x,y) and g.has_edge(x,z) and g.has_edge(y,z):
            vertex_profile[z].num_wedge_center += 1
            vertex_profile[x].num_wedge_leaf += 1
            vertex_profile[y].num_wedge_leaf += 1
        elif g.has_edge(x,y) and not g.has_edge(x,z) and not g.has_edge(y,z):
            vertex_profile[x].num_disc_leaf += 1
            vertex_profile[y].num_disc_leaf += 1
            vertex_profile[z].num_disc_alone += 1
        elif not g.has_edge(x,y) and g.has_edge(x,z) and not g.has_edge(y,z):
            vertex_profile[x].num_disc_leaf += 1
            vertex_profile[z].num_disc_leaf += 1
            vertex_profile[y].num_disc_alone += 1
        elif not g.has_edge(x,y) and not g.has_edge(x,z) and g.has_edge(y,z):
            vertex_profile[y].num_disc_leaf += 1
            vertex_profile[z].num_disc_leaf += 1
            vertex_profile[x].num_disc_alone += 1
        elif not g.has_edge(x,y) and not g.has_edge(x,z) and not g.has_edge(y,z):
            vertex_profile[y].num_anti_triangle += 1
            vertex_profile[z].num_anti_triangle += 1
            vertex_profile[x].num_anti_triangle += 1
        else:
            print('missing on ',(x,y,z))

    for v in g.nodes:
        vertex_profile[v].num_wedges = vertex_profile[v].num_wedge_center + vertex_profile[v].num_wedge_leaf
        vertex_profile[v].num_discs = vertex_profile[v].num_disc_alone + vertex_profile[v].num_disc_leaf
    
    return vertex_profile

### Four Profile Helper Functions

In [3]:
def two_hop_histogram(g: nx.Graph):
    """For each node v in g, compute the number 2-paths from v to the other nodes of g

    Assumes that g is a simple graph with no loops.
    Returns a dictionary, keys are the nodes of g, and values are another dictionary that counts the number of 2-paths
    to other nodes. For example, two_hop_profile[v][u] is equal to the number of 2 paths from v to u.
    """
    two_hop_profile = {}

    neighbor_sets = {}
    # initialize tuples for each vertex
    for v in g.nodes:
        neighbor_sets[v] = set(g.neighbors(v))
        two_hop_profile[v] = dict()
        for u in g.nodes:
            two_hop_profile[v][u] = 0

            # below approach works well if instead of a dictionary, the 2-path counts were a vector
            # when a vector, it is easy to add just add two vectors together instead of neighbour checks in the next loop
            # if u in neighbor_sets[v]:
            #     two_hop_profile[v][u] = 1
            # else:
            #     two_hop_profile[v][u] = 0


    for e in g.edges:
        u = e[0]
        v = e[1]

        # for every neighbour a of v, if ua is not an edge, then uva is a 2-path
        # then, for every neighbour a of v that is not a neighbour of u, increase ua path count by 1
        for a in neighbor_sets[v].difference(neighbor_sets[u], {u}):
            two_hop_profile[u][a] += 1

        # same goes for u
        for a in neighbor_sets[u].difference(neighbor_sets[v], {v}):
            two_hop_profile[v][a] += 1

    return two_hop_profile


class vertex_3_data:
    __slots__ = ('num_triangles', 'num_wedges', 'num_discs', 'num_anti_triangle', 'num_wedge_center', 'num_wedge_leaf',
                 'num_disc_leaf', 'num_disc_alone',
                 'n1', 'n2c', 'n2e', 'n3',
                 'n1_double', 'n2c_double', 'n2e_double', 'n3_double',
                 'n1_n2c', 'n1_n2e', 'n1_n3', 'n2c_n2e', 'n2c_n3', 'n2e_n3',
                 'triangle_pairs'
                 )

    def __init__(self):
        self.num_triangles = 0
        self.num_wedges = 0
        self.num_discs = 0
        self.num_anti_triangle = 0

        self.num_wedge_center = 0
        self.num_wedge_leaf = 0
        self.num_disc_leaf = 0
        self.num_disc_alone = 0

        # 3-profile algebraic variables
        self.n1 = 0 # number of disc leafs
        self.n2c = 0 # number of wedge center (scaled by 2)
        self.n2e = 0 # number of wedge leafs
        self.n3 = 0 # number of triangles (scaled by 2)

        # 4-profile equation variables
        self.n1_double = 0
        self.n2c_double = 0
        self.n2e_double = 0
        self.n3_double = 0
        self.n1_n2c = 0
        self.n1_n2e = 0
        self.n1_n3 = 0
        self.n2c_n2e = 0
        self.n2c_n3 = 0
        self.n2e_n3 = 0

        self.triangle_pairs = set() # set of 2-tuples that form triangles with v


def three_profile_vertex(g: nx.Graph):
    """Computes the 3-profile of a graph g, the number of subgraphs of each graph on 3 vertices.

    For each vertex v, three_profile_vertex computes the number of each 3 node graph v is in.
    For example, three_profile_vertex counts the number of different triangles contained in each vertex.

    Returns a dict where the keys are the nodes of g, and the values are vertex_3_data objects, that contain the counts of different 3-vertex subgraphs.
    """
    vertex_profile = {}

    neighbor_sets = {}
    for v in g.nodes:
        neighbor_sets[v] = set(g.neighbors(v))
        g.nodes[v]['3_profile'] = vertex_3_data()

    for e in g.edges:
        u = e[0]
        v = e[1]
        num_u_neighbors = len(neighbor_sets[u])
        num_v_neighbors = len(neighbor_sets[v])
        common_neighbors = neighbor_sets[u].intersection(neighbor_sets[v])
        n3_a = len(common_neighbors)

        g.nodes[v]['3_profile'].n3 += n3_a
        g.nodes[u]['3_profile'].n3 += n3_a

        g.nodes[v]['3_profile'].n3_double += (n3_a * (n3_a - 1)) / 2
        g.nodes[u]['3_profile'].n3_double += (n3_a * (n3_a - 1)) / 2

        for neighbor in common_neighbors:
            g.nodes[v]['3_profile'].triangle_pairs.add(tuple(sorted((u,neighbor)))) # we sort since each triangle will be added twice, triangle auv will be added for u on edge au and uv
            g.nodes[u]['3_profile'].triangle_pairs.add(tuple(sorted((v,neighbor))))

        # to count wedges, split up wedges centered at u and centered at v
        # to count wedges centered at u containing uv, count all neighbors of u, except for v, and subtract common neighbors
        n2c_ua = num_u_neighbors - n3_a - 1
        g.nodes[u]['3_profile'].n2c += n2c_ua
        g.nodes[v]['3_profile'].n2e += n2c_ua

        g.nodes[u]['3_profile'].n2c_double += (n2c_ua * (n2c_ua - 1)) / 2
        g.nodes[v]['3_profile'].n2e_double += (n2c_ua * (n2c_ua - 1)) / 2

        # to count wedges centered at v containing uv, count all neighbors of v, except for u, and subtract common neighbors
        n2c_va = num_v_neighbors - n3_a - 1
        g.nodes[v]['3_profile'].n2c += n2c_va
        g.nodes[u]['3_profile'].n2e += n2c_va

        g.nodes[v]['3_profile'].n2c_double += (n2c_va * (n2c_va - 1)) / 2
        g.nodes[u]['3_profile'].n2e_double += (n2c_va * (n2c_va - 1)) / 2

        # to get disc counts, number of nodes that aren't adjacent to u or v. |V(g)| - num of distinct neighbors of u union distinct neighbors of v
        # to calculate number of distinct neighbors, take both neighbor sets combined, and substract the common ones (they're counted twice)
        n1_a = g.number_of_nodes() - num_u_neighbors - num_v_neighbors + n3_a
        g.nodes[v]['3_profile'].n1 += n1_a
        g.nodes[u]['3_profile'].n1 += n1_a

        g.nodes[v]['3_profile'].n1_double += (n1_a * (n1_a - 1)) / 2
        g.nodes[u]['3_profile'].n1_double += (n1_a * (n1_a - 1)) / 2

        # compute calculated variable for 4-profile
        g.nodes[u]['3_profile'].n1_n2c += n1_a * n2c_ua
        g.nodes[u]['3_profile'].n1_n2e += n1_a * n2c_va
        g.nodes[u]['3_profile'].n1_n3 += n1_a * n3_a
        g.nodes[u]['3_profile'].n2c_n2e += n2c_ua * n2c_va
        g.nodes[u]['3_profile'].n2c_n3 += n2c_ua * n3_a
        g.nodes[u]['3_profile'].n2e_n3 += n2c_va * n3_a

        g.nodes[v]['3_profile'].n1_n2c += n1_a * n2c_va
        g.nodes[v]['3_profile'].n1_n2e += n1_a * n2c_ua
        g.nodes[v]['3_profile'].n1_n3 += n1_a * n3_a
        g.nodes[v]['3_profile'].n2c_n2e += n2c_ua * n2c_va
        g.nodes[v]['3_profile'].n2c_n3 += n2c_va * n3_a
        g.nodes[v]['3_profile'].n2e_n3 += n2c_ua * n3_a


        # for vertex in e:
            # g.nodes[vertex]['3_profile'].n1_double += (n1_additions * (n1_additions - 1)) / 2
            # g.nodes[vertex]['3_profile'].n2c_double += (g.nodes[vertex]['3_profile'].n2c * g.nodes[vertex][
            #     '3_profile'].n2c - 1) / 2
            # g.nodes[vertex]['3_profile'].n2e_double += (g.nodes[vertex]['3_profile'].n2e * g.nodes[vertex][
            #     '3_profile'].n2e - 1) / 2
            # g.nodes[vertex]['3_profile'].n3_double += (g.nodes[vertex]['3_profile'].n3 * g.nodes[vertex][
                # '3_profile'].n3 - 1) / 2
            #
            # g.nodes[vertex]['3_profile'].n1_n2c += g.nodes[vertex]['3_profile'].n1 * g.nodes[vertex]['3_profile'].n2c
            # g.nodes[vertex]['3_profile'].n1_n2e += g.nodes[vertex]['3_profile'].n1 * g.nodes[vertex]['3_profile'].n2e
            # g.nodes[vertex]['3_profile'].n1_n3 += g.nodes[vertex]['3_profile'].n1 * g.nodes[vertex]['3_profile'].n3
            # g.nodes[vertex]['3_profile'].n2c_n2e += g.nodes[vertex]['3_profile'].n2c * g.nodes[vertex]['3_profile'].n2e
            # g.nodes[vertex]['3_profile'].n2c_n3 += g.nodes[vertex]['3_profile'].n2c * g.nodes[vertex]['3_profile'].n3
            # g.nodes[vertex]['3_profile'].n2e_n3 += g.nodes[vertex]['3_profile'].n2e * g.nodes[vertex]['3_profile'].n3

    for v in g.nodes:
        g.nodes[v]['3_profile'].num_triangles = g.nodes[v]['3_profile'].n3 / 2
        g.nodes[v]['3_profile'].num_wedges = g.nodes[v]['3_profile'].n2e + g.nodes[v][
            '3_profile'].n2c / 2

        g.nodes[v]['3_profile'].num_disc_alone = g.number_of_edges() - g.nodes[v]['3_profile'].num_triangles - \
                                                 g.nodes[v]['3_profile'].n2e - g.degree(v)
        g.nodes[v]['3_profile'].num_discs = g.nodes[v]['3_profile'].n1 + g.nodes[v][
            '3_profile'].num_disc_alone

        g.nodes[v]['3_profile'].num_anti_triangle = (g.number_of_nodes() - 1) * (g.number_of_nodes() - 2) / 2 - \
                                                    g.nodes[v]['3_profile'].num_triangles - g.nodes[v][
                                                        '3_profile'].num_wedges - g.nodes[v]['3_profile'].num_discs

        vertex_profile[v] = g.nodes[v]['3_profile']

    return vertex_profile

### Four Profile

In [4]:
def four_profile(g: nx.Graph):
    """Computes the number of 4-node subgraphs that exists in a NetworkX Graph

    This is non-distributed implementation of algorithm from the paper
    "Distributed Estimation of Graph 4-Profiles" by E. Elenberg et al, 2016


    Returns a numpy vector of length 11, where component i is the count of F_i
    """
    two_hop_profile = two_hop_histogram(g)
    three_profile = three_profile_vertex(g)

    neighbor_sets = {}
    for v in g.nodes:
        neighbor_sets[v] = set(g.neighbors(v))
        g.nodes[v]['3_profile'] = vertex_3_data()

    four_profile_global = np.zeros(11)
    four_profile_local_equation = {}
    for v in g.nodes:
        four_profile_equation_data = np.zeros(17)
        four_profile_equation_data[0] = three_profile[v].n1_double
        four_profile_equation_data[1] = three_profile[v].n2c_double
        four_profile_equation_data[2] = three_profile[v].n2e_double
        four_profile_equation_data[3] = three_profile[v].n3_double
        four_profile_equation_data[4] = three_profile[v].n1_n2c
        four_profile_equation_data[5] = three_profile[v].n1_n2e
        four_profile_equation_data[6] = three_profile[v].n1_n3
        four_profile_equation_data[7] = three_profile[v].n2c_n2e
        four_profile_equation_data[8] = three_profile[v].n2c_n3
        four_profile_equation_data[9] = three_profile[v].n2e_n3
        four_profile_equation_data[10] = three_profile[v].num_disc_alone * g.degree(v)

        # (number of 2-paths to non-neighbours) choose 2
        four_profile_equation_data[11] = 0
        for a in set(g.nodes).difference(neighbor_sets[v], {v}):
            four_profile_equation_data[11] += (two_hop_profile[v][a] * (two_hop_profile[v][a] - 1)) / 2

        # Equation for F10 in the paper
        # count number of 4-cliques containing v
        # for each neighbour a of v, count number of triangles of abc where b and c are neighbours of v
        for a in neighbor_sets[v]:
            for (b,c) in three_profile[a].triangle_pairs:
                if b in neighbor_sets[v] and c in neighbor_sets[v]:
                    four_profile_equation_data[12] += 1
        four_profile_equation_data[12] = four_profile_equation_data[12] * 2
        # each clique at a vertex is counted three times once each for every incident edge, until fix matrix just scale this to 6x

        # 4th equation in (3) in the paper
        # For each neighbour a, sum (n3a - n3va), n3a is num triangles of a
        # n3va is number of shared triangles between v and a
        # sum of n3va will be total number of triangles containing v * 2
        for a in neighbor_sets[v]:
            four_profile_equation_data[13] += three_profile[a].num_triangles
        four_profile_equation_data[13] -= three_profile[v].n3

        # 5th equation in (3) in the paper
        # For each neighbour a, sum (n2e_a - n2c_va)
        for a in neighbor_sets[v]:
            four_profile_equation_data[14] += three_profile[a].n2e
        four_profile_equation_data[14] -= three_profile[v].n2c

        # Equation for F8 in the paper
        # for each neighbour a of v, count number of triangles abc where b and c are not neighbours of v
        # b and c must also be separate from v
        for a in neighbor_sets[v]:
            for (b,c) in three_profile[a].triangle_pairs:
                if b not in neighbor_sets[v] and c not in neighbor_sets[v] and b != v and c != v:
                    four_profile_equation_data[15] += 1

        four_profile_equation_data[16] = (g.number_of_nodes() - 1) * (g.number_of_nodes() - 2) * (g.number_of_nodes() - 3) / 6 # |V|-1 choose 3

        # solving for four-profile
        A0 = np.array([-6, -2, -6, -2, -3, -6, -3, -3, -2, -3, 0, 0, 0, 0, 0, 0, 6])
        A1 = np.array([1, 0, 0, -2, 0, 0, 0, 0, 0, -1, -1, -2, 0, 2, 1, -1, 0])
        A2 = np.array([0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 1, 2, 0, -2, -1, 1, 0])
        A3 = np.array([0, 0, 0, 2, 0, 1, 0, 0, 0, 1, 0, 2, 0, -2, -1, 2, 0])
        A4 = np.array([0, 0, 0, 0, 2, 0, 0, -2, 0, 0, 0, 4, 1, -2, 0, 2, 0])
        A5 = np.array([0, 0, 0, -2, 0, 0, 0, 0, 0, -1, 0, -2, 0, 2, 1, -2, 0])
        A6 = np.array([0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, -4, -1, 2, 0, -2, 0])
        A7 = np.array([0, 0, 0, 0, 0, 0, 2, 0, 0, -2, 0, 0, -1, 2, 0, -2, 0])
        A8 = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0])
        A9 = np.array([0, 2, 0, 2, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0, 0, 0])
        A10 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 1, -2, 0, 2, 0])
        A11 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0])
        A12 = np.array([0, 0, 0, -2, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0])
        A13 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, -2, 0, 2, 0])
        A14 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 2, 0, -2, 0])
        A15 = np.array([0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0])
        A16 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

        four_profile_local = np.zeros(17)
        four_profile_local[0] = np.dot(four_profile_equation_data, A0) / 6.
        four_profile_local[1] = np.dot(four_profile_equation_data, A1)
        four_profile_local[2] = np.dot(four_profile_equation_data, A2)
        four_profile_local[3] = np.dot(four_profile_equation_data, A3)
        four_profile_local[4] = np.dot(four_profile_equation_data, A4) / 4.
        four_profile_local[5] = np.dot(four_profile_equation_data, A5)
        four_profile_local[6] = np.dot(four_profile_equation_data, A6) / 2.
        four_profile_local[7] = np.dot(four_profile_equation_data, A7) / 4.
        four_profile_local[8] = np.dot(four_profile_equation_data, A8)
        four_profile_local[9] = np.dot(four_profile_equation_data, A9) / 6.
        four_profile_local[10] = np.dot(four_profile_equation_data, A10) / 4.
        four_profile_local[11] = np.dot(four_profile_equation_data, A11)
        four_profile_local[12] = np.dot(four_profile_equation_data, A12) / 2.
        four_profile_local[13] = np.dot(four_profile_equation_data, A13) / 2.
        four_profile_local[14] = np.dot(four_profile_equation_data, A14) / 4.
        four_profile_local[15] = np.dot(four_profile_equation_data, A15) / 2.
        four_profile_local[16] = np.dot(four_profile_equation_data, A16) / 6.

        four_profile_local_equation[v] = four_profile_local
        #debugging
        # print(four_profile_local[0])

        four_profile_global[0] += four_profile_local[0] / 4.
        four_profile_global[1] += four_profile_local[1] / 2.
        four_profile_global[2] += four_profile_local[2] / 4.
        four_profile_global[3] += four_profile_local[4]
        four_profile_global[4] += four_profile_local[6] / 2.
        four_profile_global[5] += four_profile_local[7] / 3.
        four_profile_global[6] += four_profile_local[9]
        four_profile_global[7] += four_profile_local[10] / 4.
        four_profile_global[8] += four_profile_local[11]
        four_profile_global[9] += four_profile_local[14] / 2.
        four_profile_global[10] += four_profile_local[16] / 4.

    # currently N0 is not being computed correct (A0 is off)
    # Since everything else is being counted correctly, count N0 from |V| choose 4 minus all other subgraph counts
    N0 = (g.number_of_nodes() * (g.number_of_nodes() - 1) * (g.number_of_nodes() - 2) * (g.number_of_nodes() - 3)) / 24
    for i in range(1,11):
        N0 -= four_profile_global[i]

    four_profile_global[0] = N0

    return four_profile_global

### Four Profile Primitive

In [5]:
def four_profile_primitive(g: nx.Graph):
    num_clique = 0
    num_five_edge = 0
    num_square = 0
    num_spoon = 0 # triangle with a handle
    num_triangle_dot = 0 # triangle with an extra disconnected node
    num_four_path = 0
    num_claw = 0
    num_disc_path = 0 # 2 disconnected paths
    num_three_path_dot = 0 # P3 with a disconnected node
    num_one_edge = 0
    num_zero_edge = 0

    for a,b,c,d in (itertools.combinations(g.nodes, 4)):
        degrees = []
        h = nx.induced_subgraph(g, [a,b,c,d])
        degrees.append(h.degree(a))
        degrees.append(h.degree(b))
        degrees.append(h.degree(c))
        degrees.append(h.degree(d))
        
        num_edges = 0
        if g.has_edge(a,b):
            num_edges += 1
        if g.has_edge(a,c):
            num_edges += 1
        if g.has_edge(a,d):
            num_edges += 1
        if g.has_edge(b,c):
            num_edges += 1
        if g.has_edge(b,d):
            num_edges += 1
        if g.has_edge(c,d):
            num_edges += 1

        if num_edges == 6:
            num_clique += 1
        elif num_edges == 5:
            num_five_edge += 1
        elif num_edges == 4:
            if max(degrees) == 3:
                num_spoon += 1
            elif max(degrees) == 2:
                num_square += 1
            else:
                print('error on 4 edges', (a,b,c,d))
        elif num_edges == 3:
            if max(degrees) == 3:
                num_claw += 1
            elif max(degrees) == 2 and min(degrees) == 0:
                num_triangle_dot += 1
            elif max(degrees) == 2 and min(degrees) == 1:
                num_four_path += 1
            else:
                print('error on 3 edges', (a,b,c,d))
        elif num_edges == 2:
            if max(degrees) == 2:
                num_three_path_dot += 1
            elif max(degrees) == 1:
                num_disc_path += 1
            else:
                print('error on 2 edges', (a,b,c,d))
        elif num_edges == 1:
            num_one_edge += 1
        elif num_edges == 0:
            num_zero_edge += 1
        else:
            print('error on finding edges', (a,b,c,d))

    four_profile = {}
    four_profile[0] = num_zero_edge
    four_profile[1] = num_one_edge
    four_profile[2] = (num_disc_path, num_three_path_dot)
    four_profile[3] = (num_four_path, num_triangle_dot, num_claw)
    four_profile[4] = (num_square, num_spoon)
    four_profile[5] = num_five_edge
    four_profile[6] = num_clique

    return four_profile

In [6]:
gnp50 = nx.gnp_random_graph(50, 0.3, seed=3)
start = time.time()
fpfp = four_profile_primitive(gnp50)
end = time.time()

print(end-start)
fpfp

6.194712162017822


{0: 27236,
 1: 69722,
 2: (14827, 59393),
 3: (25125, 8647, 8623),
 4: (2656, 11411),
 5: 2471,
 6: 189}

In [7]:
start = time.time()
fpf = four_profile(gnp50)
end = time.time()

print(end-start)
fpf

0.013701200485229492


array([27236., 69722., 14827., 59393., 25125.,  8647.,  8623.,  2656.,
       11411.,  2471.,   189.])

In [ ]:
gnp150 = nx.gnp_random_graph(150, 0.3, seed=4)

start = time.time()
fpfp_gnp150 = four_profile_primitive(gnp150)
end = time.time()

print(end-start)
fpfp_gnp150

In [8]:
gnp150 = nx.gnp_random_graph(150, 0.3, seed=4)

start = time.time()
fpf_gnp150 = four_profile(gnp150)
end = time.time()

print(end-start)
fpf_gnp150

0.518744707107544


array([2427113., 6184880., 1316244., 5241446., 2218237.,  746781.,
        734998.,  232698.,  943607.,  200048.,   14223.])

In [15]:
g_pete = nx.petersen_graph()
fpf_pete = four_profile_primitive(g_pete)
fpf_pete

{0: 5, 1: 60, 2: (15, 60), 3: (60, 0, 10), 4: (0, 0), 5: 0, 6: 0}

In [16]:
# c6 = nx.gnp_random_graph(6, 1, seed=3)
c6 = nx.cycle_graph(6)
fpfp_c6 = four_profile_primitive(c6)
fpfp_c6

{0: 0, 1: 0, 2: (3, 6), 3: (6, 0, 0), 4: (0, 0), 5: 0, 6: 0}